In [8]:
import sys
from pathlib import Path

try:
    # 일반 Python 스크립트 실행 시 (__file__이 존재)
    current_path = Path(__file__).resolve()
except NameError:
    # Jupyter Notebook 실행 시 (__file__ 없음)
    current_path = Path().resolve()

# 현재 경로에서 stock_forecast 폴더까지 자동 탐색
for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"
        break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")

# sys.path에 추가
if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))

print(f"sys.path에 등록된 경로: {stock_forecast_path}")

sys.path에 등록된 경로: C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast


In [2]:
import pandas as pd
import numpy as np
import requests
import xmltodict
import json
import traceback
from pandas.tseries.offsets import MonthEnd
from DATA.stock_invest_function import *
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
from sqlalchemy import create_engine
import pymysql
from tqdm import tqdm

import socket

In [3]:
# f1 = Path(r"C:\Users\Hoyoung_Park\Downloads\korea_toptier_company_hscode_block1_20210924.xlsx")
# f2 = Path(r"C:\Users\Hoyoung_Park\Downloads\korea_toptier_company_hscode_block2_20210924.xlsx")
#
# print("f1 exists:", f1.exists(), "|", f1)
# print("f2 exists:", f2.exists(), "|", f2)
#
# # 혹시 다른 Downloads(OneDrive 동기화 폴더)인지도 체크
# f1_od = Path.home() / "OneDrive" / "Downloads" / f1.name
# f2_od = Path.home() / "OneDrive" / "Downloads" / f2.name
# print("OneDrive Downloads f1:", f1_od.exists(), "|", f1_od)
# print("OneDrive Downloads f2:", f2_od.exists(), "|", f2_od)
#
# # 참고: 현재 계정·작업디렉터리 표시
# print("USERPROFILE:", os.environ.get("USERPROFILE"))
# print("CWD:", Path().resolve())

In [4]:
path1 = r"C:\Users\82108\Downloads\korea_toptier_company_hscode_block1_20210924.xlsx"
path2 = r"C:\Users\82108\Downloads\korea_toptier_company_hscode_block2_20210924..xlsx"
# "C:\Users\Hoyoung_Park\Downloads\korea_toptier_company_hscode_block1_20210924.xlsx.xlsx"
df1 = pd.read_excel(path1)
df2 = pd.read_excel(path2)

df1 = df1.rename(columns={'HS_Code': 'hs_code', 'Item' : "item", "Code": "ticker"})
df2 = df2.rename(columns={'Code': 'ticker'})

In [5]:
hscode_data = pd.concat([df1, df2], join='inner')

# # Code, Name, hs_code가 모두 같은 행에서 첫 번째만 남기고 제거
# df_unique = hscode_data.drop_duplicates(subset=['Code', 'Name', 'hs_code'], keep='first')

# hs_code_df = df_unique['hs_code']

In [6]:
hscode_dat_drop = hscode_data.drop_duplicates(subset=['ticker', 'Name', 'hs_code'], keep=False)

hscode_dat_drop

,ticker,Name,hs_code
0,A093370,후성,854321
1,A036490,SK머티리얼즈,281290
2,A036490,SK머티리얼즈,8542
3,A104830,원익머트리얼즈,854239
4,A144960,뉴파워프라즈마,854239
...,...,...,...
404,A011784,금호석유,4002590000
405,A011785,금호석유,4002110000
406,A178920,PI첨단소재,3916909000
407,A000070,삼양홀딩스,290723


In [8]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',           # 예: 'root'
    'password': 'Apt106503!~',   # 예: '1234'
    'host': get_db_host(),           # 예: 'localhost'
    'port': '3307',                # 예: '3306'
    'database': 'investar'    # 예: 'trade_data'

}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 테이블 이름 지정
table_name = 'korea_company_hscode_map'

# DB에 업로드 (기존 테이블 덮어씀 → 'replace', 추가는 'append')
hscode_dat_drop.to_sql(name=table_name, con=engine, index=False, if_exists='replace')

print(f"✅ {table_name} 테이블에 {len(hscode_dat_drop)}개의 행이 업로드되었습니다.")

✅ korea_company_hscode_map 테이블에 762개의 행이 업로드되었습니다.


In [13]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',           # 예: 'root'
    'password': 'Apt106503!~',   # 예: '1234'
    'host': get_db_host(),           # 예: 'localhost'
    'port': '3307',                # 예: '3306'
    'database': 'investar'    # 예: 'trade_data'

}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 테이블 이름 지정
table_name = 'target_hs_code'

# DB에 업로드 (기존 테이블 덮어씀 → 'replace', 추가는 'append')
hs_code_df.to_sql(name=table_name, con=engine, index=False, if_exists='replace')

print(f"✅ {table_name} 테이블에 {len(hs_code_df)}개의 행이 업로드되었습니다.")

✅ target_hs_code 테이블에 808개의 행이 업로드되었습니다.


In [14]:
df_unique

,Code,Name,hs_code
0,A000080,하이트진로,2208904000
1,A000210,DL,39022
2,A000210,DL,3901101000
3,A000660,SK하이닉스,8542321010
4,A000660,SK하이닉스,8542321030
...,...,...,...
446,A020000,한섬,6103
447,A005390,신성통상,6202
448,A008290,원풍물산,6203
449,A105630,한세실업,6102


In [16]:
# SQLAlchemy 연결 URL 생성
db_url = f"mysql+pymysql://{db_info['user']}:{db_info['password']}@" \
         f"{db_info['host']}:{db_info['port']}/{db_info['database']}"

# DB 엔진 생성
engine = create_engine(db_url)

# 테이블 저장
df_unique.to_sql(
    name="target_company_db",   # 테이블명
    con=engine,
    if_exists="replace",        # 기존 테이블 있으면 덮어쓰기, append 로 변경 가능
    index=False                 # DataFrame index는 저장 안 함
)

print("데이터가 DB에 저장되었습니다.")

데이터가 DB에 저장되었습니다.
